In [ ]:
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import time
from collections import Counter

# 🔑 Your Semantic Scholar API key
API_KEY = "YOUR_API_KEY_HERE"

# File containing paper IDs (one per line)
IDS_FILE = "anchor_pids.txt"

# Read IDs from txt file
with open(IDS_FILE, "r") as f:
    paper_ids = [line.strip() for line in f if line.strip()]

print(f"📄 Loaded {len(paper_ids)} paper IDs from {IDS_FILE}")

# API configuration
BASE_URL = "https://api.semanticscholar.org/graph/v1/paper/"
HEADERS = {"x-api-key": API_KEY}

FIELDS = [
    "title",
    "abstract",
    "venue",
    "year",
    "publicationDate",
    "citationCount",
    "influentialCitationCount",
    "referenceCount",
    "fieldsOfStudy",
    "publicationTypes",
    "isOpenAccess",
    "openAccessPdf",
    "authors",
    "tldr",
    "url"
]

def fetch_paper_info(paper_id):
    url = f"{BASE_URL}{paper_id}"
    params = {"fields": ",".join(FIELDS)}
    response = requests.get(url, headers=HEADERS, params=params)
    if response.status_code == 200:
        data = response.json()
        # Flatten authors
        data["authors"] = [a.get("name", "") for a in data.get("authors", [])]
        # Flatten fields
        if "fieldsOfStudy" in data and isinstance(data["fieldsOfStudy"], list):
            data["fieldsOfStudy"] = "; ".join(data["fieldsOfStudy"])
        return data
    else:
        print(f"⚠️ Error fetching {paper_id}: {response.status_code} {response.text}")
        return None

# Fetch results
results = []
for i, pid in enumerate(paper_ids, 1):
    info = fetch_paper_info(pid)
    if info:
        results.append(info)
    print(f"Fetched {i}/{len(paper_ids)}")
    time.sleep(0.2)  # polite delay

# Convert to DataFrame
df = pd.DataFrame(results)
df.to_csv("anchor_papers_metadata.csv", index=False)
print(f"\n✅ Metadata saved to anchor_papers_metadata.csv")

# --- 📊 Summary stats ---
print("\n=== Anchor Paper Summary ===")
print(f"Total papers: {len(df)}")
print(f"Year range: {df['year'].min()} – {df['year'].max()}")
print(f"Median year: {df['year'].median()} | Mean year: {df['year'].mean():.1f}")
print(f"Citation range: {df['citationCount'].min()} – {df['citationCount'].max()}")
print(f"Median citations: {df['citationCount'].median()} | Mean: {df['citationCount'].mean():.1f}")

# --- 📊 Plots ---
plt.style.use("seaborn-v0_8")

# 1. Histogram of citation counts
plt.figure(figsize=(6,4))
plt.hist(df["citationCount"].dropna(), bins=15, edgecolor="black")
plt.xlabel("Citation Count")
plt.ylabel("Number of Papers")
plt.title("Distribution of Citation Counts (Anchor Papers)")
plt.tight_layout()
plt.savefig("anchor_citations_hist.png")
plt.close()

# 2. Histogram of publication years
plt.figure(figsize=(6,4))
plt.hist(df["year"].dropna(), bins=range(int(df["year"].min()), int(df["year"].max())+1), edgecolor="black")
plt.xlabel("Publication Year")
plt.ylabel("Number of Papers")
plt.title("Distribution of Publication Years (Anchor Papers)")
plt.tight_layout()
plt.savefig("anchor_years_hist.png")
plt.close()

# 3. Bar chart of fields of study
fields = []
for f in df["fieldsOfStudy"].dropna():
    fields.extend([x.strip() for x in f.split(";")])
field_counts = Counter(fields).most_common(10)

plt.figure(figsize=(7,4))
plt.bar([f[0] for f in field_counts], [f[1] for f in field_counts])
plt.xticks(rotation=45, ha="right")
plt.ylabel("Count")
plt.title("Top Fields of Study (Anchor Papers)")
plt.tight_layout()
plt.savefig("anchor_fields_bar.png")
plt.close()

# 4. Bar chart of venues
venue_counts = Counter(df["venue"].dropna()).most_common(10)
plt.figure(figsize=(7,4))
plt.bar([v[0] for v in venue_counts], [v[1] for v in venue_counts])
plt.xticks(rotation=45, ha="right")
plt.ylabel("Count")
plt.title("Top Venues (Anchor Papers)")
plt.tight_layout()
plt.savefig("anchor_venues_bar.png")
plt.close()

print("✅ Plots saved: anchor_citations_hist.png, anchor_years_hist.png, anchor_fields_bar.png, anchor_venues_bar.png")


📄 Loaded 20 paper IDs from anchor_pids.txt
Fetched 1/20
Fetched 2/20
Fetched 3/20
Fetched 4/20
Fetched 5/20
Fetched 6/20
Fetched 7/20
Fetched 8/20
Fetched 9/20
Fetched 10/20
Fetched 11/20
Fetched 12/20
Fetched 13/20
Fetched 14/20
Fetched 15/20
Fetched 16/20
Fetched 17/20
Fetched 18/20
Fetched 19/20
Fetched 20/20

✅ Metadata saved to anchor_papers_metadata.csv

=== Anchor Paper Summary ===
Total papers: 20
Year range: 2017 – 2024
Median year: 2021.5 | Mean year: 2021.5
Citation range: 17 – 633
Median citations: 174.5 | Mean: 217.5
✅ Plots saved: anchor_citations_hist.png, anchor_years_hist.png, anchor_fields_bar.png, anchor_venues_bar.png


In [4]:
import matplotlib.pyplot as plt
import seaborn as sns
import seaborn as sns
sns.set_style("whitegrid")
sns.set_context("talk")

# 1. Citation counts (log scale histogram)
plt.figure(figsize=(6,4))
sns.histplot(df["citationCount"].dropna(), bins=20, log_scale=True, color="skyblue", edgecolor="black")
plt.xlabel("Citation Count (log scale)")
plt.ylabel("Number of Papers")
plt.title("Citation Distribution of Anchor Papers")
plt.tight_layout()
plt.savefig("anchor_citations.png")
plt.close()

# 2. Publication years (bar plot)
plt.figure(figsize=(6,4))
year_counts = df["year"].dropna().value_counts().sort_index()
sns.barplot(x=year_counts.index, y=year_counts.values, color="lightcoral", edgecolor="black")
plt.xlabel("Publication Year")
plt.ylabel("Number of Papers")
plt.title("Anchor Papers by Year")
plt.tight_layout()
plt.savefig("anchor_years.png")
plt.close()

# 3. Fields of study (horizontal bar chart)
fields = []
for f in df["fieldsOfStudy"].dropna():
    fields.extend([x.strip() for x in f.split(";")])
field_counts = Counter(fields).most_common(8)  # top 8

plt.figure(figsize=(7,4))
sns.barplot(y=[f[0] for f in field_counts], x=[f[1] for f in field_counts], color="mediumseagreen", edgecolor="black")
plt.xlabel("Count")
plt.ylabel("Field of Study")
plt.title("Top Fields of Study")
plt.tight_layout()
plt.savefig("anchor_fields.png")
plt.close()

# 4. Venues (horizontal bar chart)
venue_counts = Counter(df["venue"].dropna()).most_common(8)  # top 8

plt.figure(figsize=(7,4))
sns.barplot(y=[v[0] for v in venue_counts], x=[v[1] for v in venue_counts], color="goldenrod", edgecolor="black")
plt.xlabel("Count")
plt.ylabel("Venue")
plt.title("Top Venues of Anchor Papers")
plt.tight_layout()
plt.savefig("anchor_venues.png")
plt.close()

print("✅ Cleaner plots saved: anchor_citations.png, anchor_years.png, anchor_fields.png, anchor_venues.png")


✅ Cleaner plots saved: anchor_citations.png, anchor_years.png, anchor_fields.png, anchor_venues.png


/var/folders/30/nb0zdrdn59g2mzr_dg865x840000gn/T/ipykernel_37706/2200591982.py:51: UserWarning: Tight layout not applied. The left and right margins cannot be made large enough to accommodate all axes decorations.
  plt.tight_layout()


In [5]:
# Take top 10 venues (or fewer if dataset is small)
venue_counts = Counter(df["venue"].dropna()).most_common(10)

# Build LaTeX table string
latex_table = "\\begin{table}[h]\n\\centering\n\\small\n"
latex_table += "\\begin{tabular}{lc}\n\\toprule\n"
latex_table += "\\textbf{Venue} & \\textbf{Count} \\\\\n\\midrule\n"

for venue, count in venue_counts:
    latex_table += f"{venue} & {count} \\\\\n"

latex_table += "\\bottomrule\n\\end{tabular}\n"
latex_table += "\\caption{Top venues of anchor papers selected by experts.}\n"
latex_table += "\\label{tab:anchor_venues}\n\\end{table}"

# Print LaTeX table so you can copy-paste into your paper
print(latex_table)


\begin{table}[h]
\centering
\small
\begin{tabular}{lc}
\toprule
\textbf{Venue} & \textbf{Count} \\
\midrule
arXiv.org & 3 \\
Neural Information Processing Systems & 2 \\
Annual Meeting of the Association for Computational Linguistics & 2 \\
Computer Vision and Pattern Recognition & 2 \\
North American Chapter of the Association for Computational Linguistics & 2 \\
International Conference on Computer Graphics and Interactive Techniques & 1 \\
IEEE Access & 1 \\
Knowledge Discovery and Data Mining & 1 \\
ACM Transactions on Intelligent Systems and Technology & 1 \\
ACM Conference on Health, Inference, and Learning & 1 \\
\bottomrule
\end{tabular}
\caption{Top venues of anchor papers selected by experts.}
\label{tab:anchor_venues}
\end{table}


In [6]:
import json
from collections import Counter
import pandas as pd

papers = []
with open("anchor_papers_with_topics.jsonl") as f:
    for line in f:
        papers.append(json.loads(line))

df = pd.DataFrame(papers)


In [7]:
topic_counts = Counter(df["shared_topic"])
print(topic_counts)


Counter({'Graph Neural Networks and Graph Learning': 3, 'Generative Models for Images': 2, 'Recommender Systems with LLMs': 2, 'Medical and Healthcare Applications': 2, 'Differentiable Optimization': 2, 'Bias and Fairness in NLP': 2, 'Hallucination and Factuality in LLMs': 2, 'Reinforcement Learning for Scheduling': 2, 'Retrieval-Augmented Generation and Information Refinement': 1, 'Continual Learning and Knowledge Distillation': 1, 'Fake News Detection': 1})


In [8]:
latex_table = df["shared_topic"].value_counts().to_latex(
    index=True,
    header=["Count"],
    caption="Distribution of anchor papers across shared topics.",
    label="tab:anchor_topics"
)
print(latex_table)


\begin{table}
\caption{Distribution of anchor papers across shared topics.}
\label{tab:anchor_topics}
\begin{tabular}{lr}
\toprule
 & Count \\
shared_topic &  \\
\midrule
Graph Neural Networks and Graph Learning & 3 \\
Generative Models for Images & 2 \\
Recommender Systems with LLMs & 2 \\
Medical and Healthcare Applications & 2 \\
Differentiable Optimization & 2 \\
Bias and Fairness in NLP & 2 \\
Hallucination and Factuality in LLMs & 2 \\
Reinforcement Learning for Scheduling & 2 \\
Retrieval-Augmented Generation and Information Refinement & 1 \\
Continual Learning and Knowledge Distillation & 1 \\
Fake News Detection & 1 \\
\bottomrule
\end{tabular}
\end{table}

